# 15.04 · Jev in LangGraph: decision nodes and escalation

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 3 h |
| 📋 Prerequisites | Lessons 15.01–15.03; Module 07 (StateGraph, conditional edges), Module 08 (checkpointers, interrupt, Send). |
| 🔑 Keys | optional `TYPESAFE_API_KEY` · optional `GROQ_API_KEY` (the escalation model) · optional `GOOGLE_API_KEY` (Gemini as escalation) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (no network, free) · **⚖️** Jev: live with `TYPESAFE_API_KEY` (paid, fractions of a cent), otherwise replayed from recordings · **💸** calls Groq (uses your free-tier quota).
> Without keys, ⚖️ cells replay recorded answers, 💸 cells print `⏭️ Skipped`, and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** a graph where code owns the flow and Jev answers the questions at each branch.
2. **Escalate** only uncertain tickets to an LLM (Groq or Gemini): *cheap by default, frontier on exception*.
3. **Pause** for a human with `interrupt` when a decision is high-stakes, and resume from the checkpoint.
4. **Fan out** hundreds of tickets with `Send` and measure accuracy, escalation rate and cost against an all-LLM router.
5. **Explain** why checkpointing decisions matters: a resumed run must not re-decide.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Building prod with Jev and LangGraph](https://www.langchain.com/blog/building-prod-with-jev-and-langgraph) | The discovery-review graph; "cheap by default, frontier on exception" |
| [LangGraph: Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) | `StateGraph`, conditional edges, `Send`, reducers |
| [LangGraph: Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) | `interrupt`, `Command(resume=...)`, re-execution on resume |
| [LangGraph: Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | Checkpointers, threads, `get_state` |

> **Why this matters at work.** Orbita's routing has rules that live in no model: enterprise customers who sound like they might leave go to an account manager first, and uncertain tickets get a second opinion before a customer waits on the wrong queue. A graph puts those rules in **code** you can read and test. Jev makes the judgments at each branch in milliseconds, and the expensive LLM runs only on the tickets that need it.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-typesafe[experimental]==0.0.1a3" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain-typesafe", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm15-jev-in-langgraph'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from mastery.decisions import llm_decider, provenance, top_probability
from mastery.decisions.graph import build_batch_graph, build_triage_graph
from mastery.decisions.orbita import TICKETS, lab_tickets, queue_questions, ticket_state, triage_questions
from mastery.testing import ScriptedChatModel

ESCALATE_PROVIDER = "groq"      # ← or "google_genai" (Gemini free tier, needs GOOGLE_API_KEY)

---
## Part 1 · The graph: Jev at the branch, code owns the flow

`build_triage_graph` (in `mastery.decisions.graph`) has four nodes:

- **`decide`**: one Jev request, four questions (team, urgency, is_bug, churn_risk).
- **`branch`** (a conditional edge, plain Python): an enterprise ticket with `p(churn) ≥ 0.7` goes to **`account_review`**, a ticket with top probability `≥ 0.8` goes straight to **`queue`**, and anything else goes to **`escalate`**.
- **`escalate`**: an LLM answers the same questions (`llm_decider`, Lesson 15.03).
- **`account_review`**: `interrupt()` pauses the graph until a person confirms.

In [ ]:
# 🧪 The escalation model: Groq (or Gemini) with a key; offline, a scripted stand-in that answers like an LLM would.
HAVE_ESCALATION_KEY = HAVE_GROQ_KEY if ESCALATE_PROVIDER == "groq" else is_set("GOOGLE_API_KEY")
if HAVE_ESCALATION_KEY:
    from mastery.models import get_model
    escalation_model, label = get_model("fast", ESCALATE_PROVIDER, temperature=0), None
else:
    def stand_in(messages):     # reads the ticket like a careful human would, answers in the required schema
        text = str(messages[-1].content).lower()
        team = "billing" if "invoice" in text or "charged" in text else ("technical" if "slow" in text else "account_access")
        return AIMessage("", tool_calls=[{"name": "DecisionAnswers", "id": "e1",
                                          "args": {"team": team, "urgency": 1, "is_bug": 0.4, "churn_risk": 0.3}}])
    escalation_model, label = ScriptedChatModel(responses=[stand_in] * 50), "offline-stand-in"
escalate = llm_decider(escalation_model, name=label)

calls = {"jev": 0}
def counted(payload):           # count Jev calls, to prove Part 3's point about resuming
    calls["jev"] += 1
    return jev.invoke(payload)

from langchain_core.runnables import RunnableLambda
graph = build_triage_graph(RunnableLambda(counted), escalate, route_at=0.8, churn_at=0.7, questions=triage_questions,
                           state_of=ticket_state, provenance=provenance, checkpointer=InMemorySaver())
print(graph.get_graph().draw_mermaid())
print("escalation model:", "live " + ESCALATE_PROVIDER if HAVE_ESCALATION_KEY else "offline stand-in (no key)")

**You should now see** a Mermaid diagram, `__start__ → decide`, then dotted conditional edges to `queue`, `escalate` and `account_review`, all ending at `queue → __end__`, followed by the escalation model in use. Paste the diagram into [mermaid.live](https://mermaid.live) to see it drawn.

---
## Part 2 · Route the six tickets

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ (+💸 for escalations with a key) One thread per ticket; an interrupt pauses that ticket only.
rows, paused = [], {}
for t in TICKETS:
    config = {"configurable": {"thread_id": f"ticket-{t['id']}"}}
    out = graph.invoke({"ticket": t}, config)
    if "__interrupt__" in out:
        paused[t["id"]] = (config, out["__interrupt__"][0].value)
        rows.append({"id": t["id"], "queue": "⏸ waiting for a human", "by": "", "p_top": ""})
    else:
        r = out["result"]
        rows.append({"id": t["id"], "queue": r["queue"], "by": r["by"], "p_top": r["p_top"]})
table(rows, ["id", "queue", "by", "p_top"])
print("\nJev calls so far:", calls["jev"])
for tid, (_, ask) in paused.items():
    print(f"\n{tid} is paused for review:", ask)

**You should now see** T1–T4 queued by Jev, T5 escalated (`by: llm fallback`), and **T6 waiting for a human**: it's an enterprise account with `p_churn = 0.74`. The review request shows the suggested team and the question. `Jev calls so far: 6`.

One decision request per ticket, one LLM call for one ticket in six, and a human only where the business asked for one.

---
## Part 3 · The human answers, and the graph resumes without re-deciding

In [ ]:
# 🧪 Resume T6 with the account manager's verdict. The checkpoint already holds Jev's decision.
config, ask = paused["T6"]
print("next node before resuming:", graph.get_state(config).next)
out = graph.invoke(Command(resume={"team": "technical", "alert_am": True}), config)
print("result:", out["result"])
print("Jev calls after resuming:", calls["jev"], "(unchanged: decide did not run again)")

**You should now see** `next node before resuming: ('account_review',)`, a result with `by: human` and `alert_account_manager: True`, and **still 6 Jev calls**.

- **Durable execution:** the checkpointer stored `decide`'s output, so resuming starts at the interrupted node. A decision made before a pause or a crash is never made again, possibly differently.
- `interrupt()` **re-runs its node from the top** on resume. Keep side effects such as sending email *after* the interrupt, or in the next node.
- Decisions are stored in state as **plain JSON** (`model_dump`), not as Python objects, so any checkpointer (SQLite, Postgres from Module 12) can save them.

---
## Part 4 · 234 tickets with Send: what does "frontier on exception" buy?

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

The second graph fans the lab set (Module 14's 234 held-out tickets) out with **`Send`**, one branch per ticket, run in parallel. Each branch asks Jev; if the top probability is below 0.8 it escalates. To keep this offline and honest, the escalation answer is **Module 14's recorded `qwen3:4b` few-shot label** for that ticket: a real recorded model output, standing in for "the bigger model". Escalation cost is priced as one `gpt-oss-120b` call of about 400 input and 20 output tokens.

In [ ]:
# ⚖️ Fan-out over the lab set; escalations use Module 14's recorded labels (offline, real outputs).
from mastery.finetune.predictions import RECORDED_DIR, load_predictions
from mastery.pricing import estimate_cost

recorded_llm = load_predictions(RECORDED_DIR / "fewshot-qwen3-4b.json")["labels"]
LLM_CALL = estimate_cost({"input_tokens": 400, "output_tokens": 20}, "groq:openai/gpt-oss-120b")
LAB = lab_tickets()
gold = {t["id"]: t["queue"] for t in LAB}

def route_lab(ticket):
    r = jev.invoke({"state": ticket["text"], "questions": queue_questions()})
    q = r.choices["queue"]
    if top_probability(q) >= 0.8:
        return {"queue": q.choice, "by": "jev", "jev_cost": decision_cost(r) or 0.0}
    return {"queue": recorded_llm.get(ticket["id"], "error"), "by": "llm", "jev_cost": decision_cost(r) or 0.0}

from mastery.decisions import decision_cost
start = time.perf_counter()
results = build_batch_graph(route_lab).invoke({"tickets": LAB}, {"max_concurrency": 16})["results"]
seconds = time.perf_counter() - start

esc = [r for r in results if r["by"] == "llm"]
acc = sum(r["queue"] == gold[r["id"]] for r in results) / len(results)
llm_only_acc = sum(recorded_llm.get(i) == g for i, g in gold.items()) / len(gold)
hybrid_cost = sum(r["jev_cost"] for r in results) + len(esc) * LLM_CALL
llm_only_cost = len(results) * LLM_CALL
table([{"policy": "Jev, escalate < 0.8", "accuracy": f"{acc:.1%}", "LLM calls": len(esc), "$ per 1k tickets": f"{hybrid_cost / len(results) * 1000:.3f}"},
       {"policy": "LLM on every ticket", "accuracy": f"{llm_only_acc:.1%}", "LLM calls": len(results), "$ per 1k tickets": f"{llm_only_cost / len(results) * 1000:.3f}"}],
      ["policy", "accuracy", "LLM calls", "$ per 1k tickets"])
print(f"\n{len(results)} tickets in {seconds:.2f} s via Send · Jev answers: {provenance(jev.invoke({'state': LAB[0]['text'], 'questions': queue_questions()}))}")

**You should now see** the hybrid policy matching the all-LLM router's accuracy (about **94%** for both) with **27 LLM calls instead of 234**, at about a quarter of the cost per 1,000 tickets. In replay mode Jev's side is synthetic, so treat these exact numbers as a demonstration of the method. Live, the same cell measures your real trade-off.

- **Why the hybrid keeps the accuracy:** the cases Jev is unsure about are where it's most often wrong, and those are exactly the ones that go to the LLM.
- `Send` gives each ticket its own branch with its own state. LangGraph runs them in parallel (`max_concurrency` caps it), and the `results` reducer (`operator.add`) gathers them.
- Cost is dominated by escalations. Lowering the threshold saves money, and Lesson 15.02 told you what that costs in misroutes.

---
## Part 5 · Live escalation (Groq or Gemini)

In [ ]:
# 💸 One real escalation: T5 through the graph with the live escalation model (1 call, ~700 tokens).
if not HAVE_ESCALATION_KEY:
    print(f"⏭️ Skipped: no key for the escalation provider {ESCALATE_PROVIDER!r} (see the setup cell).")
else:
    config = {"configurable": {"thread_id": "live-escalation-T5"}}
    start = time.perf_counter()
    out = graph.invoke({"ticket": TICKETS[4]}, config)
    state = graph.get_state(config).values
    llm_answer = state["escalation"]["answers"]["team"]["choice"]
    print(f"T5 → {out['result']['queue']} (by {out['result']['by']}, {state['escalation']['model']}) in {time.perf_counter() - start:.2f} s")
    print(f"Jev's split was {state['decision']['answers']['team']['probabilities']}; the LLM chose {llm_answer!r}")

**You should now see** (with a key) `T5 → billing` or `account_access`, `by llm fallback` with the model name, in about 0.5–2 s, plus Jev's 52/44 split for comparison. Without a key it prints `⏭️ Skipped`.

> **Gemini as the escalation model:** set `ESCALATE_PROVIDER = "google_genai"` in the setup cell and re-run. `get_model("fast", "google_genai")` uses your free-tier Gemini key (Module 11). The graph doesn't change, because the escalation is just another `{"state", "questions"} → ClassifierResponse` Runnable.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Deserializing unregistered type … ClassifierResponse` warning | Pydantic objects stored in graph state | Store `response.model_dump(mode="json")`, rebuild with `ClassifierResponse.model_validate` (as `graph.py` does) |
| Interrupted ticket decided twice | No checkpointer, or a new `thread_id` on resume | Compile with a checkpointer; resume with the **same** config |
| Side effect ran twice after resume | It sat before `interrupt()` in the same node | Move it after the interrupt or into the next node |
| `Send` branch results overwrite each other | No reducer on the results key | `Annotated[list, operator.add]` |
| 429s during fan-out | Too many concurrent Jev requests | `{"max_concurrency": N}` in the run config; `with_rate_limit_retry` |

## ⚠️ Production notes

> ⚠️ **Production note: thresholds and rules are config.** (Parts 1, 4)
> `route_at`, `churn_at` and the escalation model are parameters of `build_triage_graph`. Load them from settings (Module 12), log them with every decision, and change them through review, not by editing a node.

> ⚠️ **Production note: an interrupt is a queue.** (Parts 2–3)
> Paused threads wait for a person, so someone has to see them. List pending threads (their `get_state(...).next`), alert when one waits too long, and choose a default if nobody answers.

## 🏋️ Exercises

1. **(Easy)** Change `churn_at` to `0.9`. Which ticket stops pausing, and where does it go instead?
2. **(Medium)** Add a `pii_check` node: after `queue`, if a new `contains_pii` question says `p ≥ 0.5`, route to a `redact` node (an LLM in production; a regex for now). What does it cost you in replay mode, and why?
3. **(Medium)** In Part 4, sweep the escalation threshold (`0.6, 0.7, 0.8, 0.9`) and print accuracy, LLM calls and $ per 1k tickets for each.
4. **(Stretch)** Swap `InMemorySaver` for the SQLite checkpointer from Module 03, pause T6, **restart the kernel**, and resume T6 from disk.

<details>
<summary><b>Show solutions</b></summary>

**1.** T6's `p_churn` is 0.74 < 0.9, so it falls to the confidence branch. Its top probability is 0.71 < 0.8, so it escalates to the LLM, and nobody alerts the account manager. That's a business decision encoded in one number.

**2.** Add the question to the `decide` request (it's nearly free there, Lesson 15.03) and an edge `queue → redact` when `p ≥ 0.5`. In replay mode the new question set has no recording, so you get a `ReplayMiss`. Either go live, or wrap `jev` with the LLM fallback from 15.03.

**3.** Make `route_lab` read a global `TH`, loop over the values, and re-run the batch graph each time. Expect accuracy to rise and cost to grow as the threshold rises, until the LLM's own mistakes cap accuracy. Pick the knee using 15.02's costs.

**4.** `SqliteSaver.from_conn_string("data/outputs/m15/graph.sqlite")` (as a context manager), same `thread_id`. After the restart, rebuild the graph and call `graph.invoke(Command(resume=...), config)`. The decision comes from disk.
</details>

## 🔎 Docs drill (5 min)

When a graph resumes after interrupt(), where does execution restart: at the interrupt() line, or at the start of the node?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [LangGraph: Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) and find how resuming works.
2. The **node re-runs from its first line**. `interrupt()` then returns the resume value instead of pausing.
3. That is why `account_review` does nothing irreversible before `interrupt()`, and why `JevToolGate` (15.05) logs only after it.

</details>

## ✅ Checkpoint

- [ ] I built a graph where Jev answers at the branch and code routes.
- [ ] I escalated only uncertain tickets, to Groq or Gemini.
- [ ] I paused a high-stakes ticket with `interrupt` and resumed it without a second Jev call.
- [ ] I fanned out 234 tickets with `Send` and compared accuracy and cost with an all-LLM router.
- [ ] Restart & Run All passes with no keys; outputs cleared before saving.

## 🔎 Research checkpoint

Read the discovery-review example in [Building prod with Jev and LangGraph](https://www.langchain.com/blog/building-prod-with-jev-and-langgraph). Write down its three questions and the route each one triggers, then map them to this lesson's nodes in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- **Code owns the flow, Jev decides at the branches**: a conditional edge reads typed answers and probabilities.
- **Cheap by default, frontier on exception**: only uncertain tickets pay for an LLM, and they're the ones Jev gets wrong most.
- `interrupt` plus a checkpointer gives human review without re-deciding. Store decisions as JSON.
- `Send` fans out thousands of decisions in parallel, and a reducer gathers them.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 15-05-middleware-routing-and-guardrails.ipynb** → [`15-05-middleware-routing-and-guardrails.ipynb`](15-05-middleware-routing-and-guardrails.ipynb).

[Module 15 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)